# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [1]:
#@title 1.1 — Install
# Local: dependencies đã có sẵn trong .venv (requirements.txt) — chỉ cài khi thiếu.
import importlib.util
missing = [m for m in ['neo4j', 'faiss', 'groq', 'openai', 'sentence_transformers', 'datasets', 'networkx']
           if importlib.util.find_spec(m) is None]
if missing:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'neo4j', 'pandas', 'numpy', 'pyarrow', 'sentence-transformers',
        'faiss-cpu', 'groq', 'openai', 'tqdm', 'networkx', 'datasets'])
print('Deps OK' if not missing else f'Installed: {missing}')


Deps OK


In [2]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

# Local run: nạp .env (trên Colab dùng userdata Secrets)
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

# Provider cho extraction/generation: 'groq' (free, TPM thấp) hoặc 'openai' (nhanh, mất phí nhỏ)
EXTRACT_PROVIDER = get_secret("EXTRACT_PROVIDER", "groq").lower()
EXTRACT_MODEL = get_secret("EXTRACT_MODEL", "") or GROQ_MODEL

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

IN_COLAB = Path("/content").exists()
WORK_DIR = "/content" if IN_COLAB else "."
DATA_PATH = f"{WORK_DIR}/hackernoon_subset.csv" if IN_COLAB else "data_local/hackernoon_subset.csv"

# Bộ golden 50 câu của giảng viên — scope: 5000 dòng đầu của dump
GOLDEN_DETAILED_PATH = "data/graphrag_golden_50_first5000_detailed.csv"

LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40


E:\VinAI\Week_5\Day 19\K4-Track3-Day19-TongDuyAn-2A202601995\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [3]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
from datasets import load_dataset

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = DATA_PATH

# Golden dataset của giảng viên đánh scope "ONLY first 5000 data rows",
# nên stream đúng 5000 dòng ĐẦU theo thứ tự gốc (không ưu tiên theo MB).
LIMIT_ROWS = 5000

if Path(OUTPUT_CSV).exists():
    print(f"Đã có {OUTPUT_CSV} — bỏ qua bước stream.")
else:
    if not HF_TOKEN:
        raise ValueError("Thiếu HF_TOKEN. Thêm Hugging Face Access Token vào Secrets/.env.")
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    print("Đang kết nối luồng dữ liệu (streaming)...")
    dataset = load_dataset(DATASET_NAME, split="train", streaming=True, token=HF_TOKEN)
    rows_written = 0
    writer = None
    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        for row in tqdm(dataset, total=LIMIT_ROWS, desc="Streaming"):
            if writer is None:
                writer = csv.DictWriter(f, fieldnames=list(row.keys()), extrasaction="ignore")
                writer.writeheader()
            writer.writerow(row)
            rows_written += 1
            if rows_written >= LIMIT_ROWS:
                break
    size_mb = Path(OUTPUT_CSV).stat().st_size / (1024 * 1024)
    print(f"✅ {OUTPUT_CSV}: {rows_written:,} rows | {size_mb:.1f} MB")


Đã có data_local/hackernoon_subset.csv — bỏ qua bước stream.


In [4]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

def reset_graph():
    # Container lab dùng riêng — xoá dữ liệu cũ để lần chạy này sạch provenance
    run_cypher("MATCH (n) DETACH DELETE n")
    print("🧹 Graph cleared.")

connect_neo4j()
setup_graph_schema()
reset_graph()

✅ Neo4j connected.
✅ Schema ready.
🧹 Graph cleared.


In [5]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def golden_evidence_rows(path):
    """Các row index (0-based, theo CSV gốc) chứa evidence của Golden Dataset."""
    if not Path(path).exists():
        return set()
    g = pd.read_csv(path)
    rows = set()
    for x in g["evidence_row_ids_0based"].dropna():
        rows.update(json.loads(x))
    return rows

def standardize_news(raw):
    raw = raw.reset_index(drop=True)
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["row_idx"] = raw.index
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    # HackerNoon dump chỉ có 'description' ngắn — ghép title vào text
    # để chunk giàu ngữ cảnh hơn cho NER/RE và retrieval
    if text_col.lower() == "description":
        has_title = df["title"].str.len() > 0
        df.loc[has_title, "text"] = (
            df.loc[has_title, "title"] + ". " + df.loc[has_title, "text"]
        ).map(norm_space)

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        # Ưu tiên giữ mọi bài báo chứa evidence của Golden Dataset,
        # phần còn lại lấy mẫu ngẫu nhiên (seed cố định) cho đủ hạn mức
        priority = golden_evidence_rows(GOLDEN_DETAILED_PATH)
        is_gold = df["row_idx"].isin(priority)
        gold_df = df[is_gold]
        rest_df = df[~is_gold]
        n_fill = max(0, LAB_MAX_ARTICLES - len(gold_df))
        fill_df = rest_df.sample(n_fill, random_state=SEED) if n_fill < len(rest_df) else rest_df
        df = pd.concat([gold_df, fill_df]).sort_values("row_idx").reset_index(drop=True)
        print(f"Article selection: {len(gold_df)} golden-evidence + {len(fill_df)} sampled")
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df, priority_rows=None):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "row_idx": r.row_idx,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
    cdf = pd.DataFrame(rows)
    if LAB_MAX_CHUNKS and len(cdf) > LAB_MAX_CHUNKS:
        # Cắt hạn mức nhưng ưu tiên giữ chunk thuộc bài báo golden-evidence
        pr = priority_rows or set()
        is_gold = cdf["row_idx"].isin(pr)
        gold = cdf[is_gold]
        rest = cdf[~is_gold].head(max(0, LAB_MAX_CHUNKS - len(gold)))
        cdf = pd.concat([gold, rest]).sort_index().reset_index(drop=True)
    return cdf

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
GOLDEN_ROWS = golden_evidence_rows(GOLDEN_DETAILED_PATH)
chunks_df = build_chunks(news_df, priority_rows=GOLDEN_ROWS)
print(f"Articles: {len(news_df)} | Chunks: {len(chunks_df)} | Golden-evidence chunks: {int(chunks_df['row_idx'].isin(GOLDEN_ROWS).sum())}")
display(chunks_df.head())

Exact dedup: 2,695 -> 2,119
Article selection: 51 golden-evidence + 1449 sampled


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

Chunking: 100%|██████████| 1500/1500 [00:00<00:00, 55925.54it/s]

Articles: 1500 | Chunks: 1500 | Golden-evidence chunks: 51


,chunk_id,article_id,row_idx,title,published_date,text
0,692168d7521764b326d8::c0000,692168d7521764b326d8,0,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications. (Nasdaq: ON) a leader in in...
1,3d9db98a718bca7cbecc::c0000,3d9db98a718bca7cbecc,2,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery. To deliver 21st-century gove...
2,16730bfb5f709838e2ba::c0000,16730bfb5f709838e2ba,3,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity. In February GreenPages acqu...
3,e0424f1d94d83c92062d::c0000,e0424f1d94d83c92062d,27,5 Kubernetes technology vendors hot right now,2023-02-28,5 Kubernetes technology vendors hot right now. Kubernetes is a technology that has created a whole new ecosystem aro...
4,a92a9be030d9548b4a25::c0000,a92a9be030d9548b4a25,28,Bachelor of Science in Health Information Management,2023-08-16,Bachelor of Science in Health Information Management. Health information management (HIM) is a diverse yet evolving ...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

openai_client = None
if EXTRACT_PROVIDER == "openai" or JUDGE_PROVIDER == "openai":
    from openai import OpenAI
    openai_client = OpenAI(api_key=OPENAI_API_KEY)

GROQ_MODEL_PREFIXES = ("openai/gpt-oss", "llama", "meta-llama", "qwen", "moonshotai")

def is_groq_model(model):
    return str(model).startswith(GROQ_MODEL_PREFIXES)

GROQ_TPM_BUDGET = 7000   # tier hiện tại: 8000 tokens/phút — chừa headroom
GROQ_STATS = {"calls": 0, "total_tokens": 0}

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=6):
    # Tên hàm giữ nguyên để tương thích các cell sau; thực tế route
    # theo model: model Groq -> Groq client (có TPM throttle),
    # ngược lại -> OpenAI client (TPM cao, không cần throttle)
    model = model or EXTRACT_MODEL
    if not model:
        raise RuntimeError("Thiếu EXTRACT_MODEL/GROQ_MODEL.")
    use_groq = is_groq_model(model)
    if use_groq and groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    if not use_groq and openai_client is None:
        raise RuntimeError("Thiếu OPENAI_API_KEY.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            if use_groq and str(model).startswith("openai/gpt-oss"):
                kwargs["reasoning_effort"] = "low"

            client = groq_client if use_groq else openai_client
            resp = client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            GROQ_STATS["calls"] += 1
            GROQ_STATS["total_tokens"] += usage.get("total_tokens") or 0
            if use_groq:
                # Throttle chủ động theo TPM (Groq free 8000 TPM) để tránh bão 429
                time.sleep(min(60, (usage.get("total_tokens") or 0) / GROQ_TPM_BUDGET * 60))
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(30, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [7]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

# Ưu tiên chunk thuộc bài báo golden-evidence vào tập extraction
gold_chunk_mask = chunks_df["row_idx"].isin(GOLDEN_ROWS)
extraction_source = pd.concat([
    chunks_df[gold_chunk_mask],
    chunks_df[~gold_chunk_mask],
]).head(EXTRACTION_MAX_CHUNKS).copy()
n_gold_in_set = int(extraction_source["row_idx"].isin(GOLDEN_ROWS).sum())
print(f"Extraction set: {len(extraction_source)} chunks ({n_gold_in_set} golden-evidence)")

COREF_CACHE = "data_local/coref_cache.csv"
coref_df = None
if Path(COREF_CACHE).exists():
    _c = pd.read_csv(COREF_CACHE)
    _c["unresolved_mentions"] = _c["unresolved_mentions"].map(json.loads)
    if set(extraction_source.chunk_id) <= set(_c.chunk_id):
        coref_df = _c[_c.chunk_id.isin(set(extraction_source.chunk_id))].reset_index(drop=True)
        print(f"Coref: dùng cache {COREF_CACHE}")

t0 = time.perf_counter()
if coref_df is None:
    coref_df = run_coref(extraction_source, batch_size=4)
    _s = coref_df.copy()
    _s["unresolved_mentions"] = _s["unresolved_mentions"].map(json.dumps)
    Path("data_local").mkdir(exist_ok=True)
    _s.to_csv(COREF_CACHE, index=False)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")
COREF_SECONDS = time.perf_counter() - t0
n_unres = sum(1 for x in extraction_source.unresolved_mentions if isinstance(x, list) and x)
print(f"Coref: {COREF_SECONDS/60:.1f} phút | chunks có unresolved mentions: {n_unres} | Groq stats: {GROQ_STATS}")

Extraction set: 400 chunks (51 golden-evidence)
Coref: dùng cache data_local/coref_cache.csv
Coref: 0.0 phút | chunks có unresolved mentions: 35 | Groq stats: {'calls': 0, 'total_tokens': 0}


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [8]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

TRIPLES_CACHE = "data_local/raw_triples_cache.csv"
TRIPLES_CACHE_META = "data_local/raw_triples_cache_meta.json"
_chunk_sig = sha1(",".join(sorted(extraction_source.chunk_id)))
raw_triples_df = None
if Path(TRIPLES_CACHE).exists() and Path(TRIPLES_CACHE_META).exists():
    _meta = json.loads(Path(TRIPLES_CACHE_META).read_text())
    if _meta.get("chunk_sig") == _chunk_sig:
        raw_triples_df = pd.read_csv(TRIPLES_CACHE).fillna({"evidence": ""})
        extraction_errors_df = pd.DataFrame()
        print(f"NER+RE: dùng cache {TRIPLES_CACHE}")

t0 = time.perf_counter()
if raw_triples_df is None:
    raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
    raw_triples_df.to_csv(TRIPLES_CACHE, index=False)
    Path(TRIPLES_CACHE_META).write_text(json.dumps({"chunk_sig": _chunk_sig}))
EXTRACT_SECONDS = time.perf_counter() - t0
print(f"NER+RE: {EXTRACT_SECONDS/60:.1f} phút | triples={len(raw_triples_df)} | batch lỗi={len(extraction_errors_df)} | Groq stats: {GROQ_STATS}")
display(raw_triples_df.head())

NER+RE: dùng cache data_local/raw_triples_cache.csv
NER+RE: 0.0 phút | triples=252 | batch lỗi=0 | Groq stats: {'calls': 0, 'total_tokens': 0}


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Aeris,Company,ACQUIRED,IoT Business from Ericsson,Technology,d1585622b7a7dccdf825::c0000,2022-12-07,Aeris to Acquire IoT Business from Ericsson.,1.0
1,Aeris,Company,PARTNERED_WITH,Ericsson,Company,d1585622b7a7dccdf825::c0000,2022-12-07,Aeris Communications and Ericsson are joining together.,1.0
2,Samsung Electronics Co. Ltd.,Company,DEVELOPED,advanced semiconductor technology,Technology,27f4e7b18f26b3718dbc::c0000,2023-10-05,"Samsung Electronics Co. Ltd., a world leader in advanced semiconductor technology.",1.0
3,DEWA,Company,USES,ChatGPT technology,Technology,0071a3ffd45898dd92f3::c0000,2023-02-09,DEWA is working to enrich its services with ChatGPT technology.,1.0
4,HE Saeed Mohammed Al Tayer,Person,WORKED_AT,DEWA,Company,0071a3ffd45898dd92f3::c0000,2023-02-09,"HE Saeed Mohammed Al Tayer, MD & CEO of Dubai Electricity and Water Authority (DEWA).",1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [9]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        AUDIT_LOG_MIN_SIM = 0.80  # log cả cặp dưới ngưỡng merge để audit minh bạch
        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                score = float(score)
                if j < 0 or i >= j or score < AUDIT_LOG_MIN_SIM:
                    continue
                ok_guard = merge_guard(names[i], names[j])
                if score >= threshold:
                    decision = "MERGE_VECTOR" if ok_guard else "REJECT_GUARD"
                else:
                    decision = "REJECT_THRESHOLD"  # chỉ ghi log, không xét merge
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": score,
                    "decision": decision
                })
                if score >= threshold and ok_guard:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
print(f"Audit rows: {len(entity_resolution_audit_df)} | decisions: {entity_resolution_audit_df.decision.value_counts().to_dict() if not entity_resolution_audit_df.empty else {}}")
print(f"Triples sau canonicalize: {len(triples_df)}")
display(entity_resolution_audit_df.head(20))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7761.09it/s]

Audit rows: 12 | decisions: {'REJECT_THRESHOLD': 7, 'MERGE_VECTOR': 4, 'REJECT_GUARD': 1}
Triples sau canonicalize: 251


,type,left,right,similarity,decision
0,Company,L&T Technology Services Limited,L&T Technology Services,0.925773,MERGE_VECTOR
1,Company,L&T Technology Services Limited,L&T Technology Services Ltd.,0.890695,REJECT_THRESHOLD
2,Company,L&T Technology Services,L&T Technology Services Ltd.,0.866094,REJECT_THRESHOLD
3,Company,General Services Administration,General Services Administration (GSA),0.831445,REJECT_THRESHOLD
4,Company,Fidelity National Information Services Inc.,Fidelity National Information Services,0.924524,MERGE_VECTOR
5,Company,Houston,Houston Texas,0.945098,REJECT_GUARD
6,Company,Stewart Information Services Corporation,Stewart Information Services,0.922792,MERGE_VECTOR
7,Company,Synergy Quantum India,Synergy Quantum,0.867454,REJECT_THRESHOLD
8,Technology,ChatGPT technology,ChatGPT,0.876033,REJECT_THRESHOLD
9,Technology,AI service,AI services,0.967647,MERGE_VECTOR


In [10]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

t0 = time.perf_counter()
nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)
print(f"Bulk insert: {time.perf_counter()-t0:.1f}s | nodes={len(nodes_df)} | edge rows={len(triples_df)}")

Bulk insert: 0.4s | nodes=382 | edge rows=251


In [11]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 382, 'edges': 251, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,fb0f4df56fab164ec48722f0,Microsoft,Company,17
1,7b29988cfc0dac3059f47a0e,L&T Technology Services,Company,7
2,cc9c6ee3857729e221d3f6de,ServiceNow,Company,6
3,773eeb9b7cc008bff365fcdd,OpenAI,Company,5
4,68b3544368862fc263941c8d,Amazon,Company,5
5,08193f3ca0efe2f71d179f4a,Renovus,Company,4
6,ce0b6a8e50a6a58526e4c85f,Aeris,Company,4
7,b823fb02e58a12bdc4c38934,NVIDIA,Company,3
8,261a323bf5773a8433c812a6,DriveNets,Company,3
9,43a2b279a035975ab78ed013,IIIT Hyderabad,Company,3


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [12]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   8%|▊         | 1/12 [00:02<00:30,  2.80s/it]

Batches:  17%|█▋        | 2/12 [00:03<00:16,  1.70s/it]

Batches:  25%|██▌       | 3/12 [00:04<00:12,  1.38s/it]

Batches:  33%|███▎      | 4/12 [00:06<00:10,  1.37s/it]

...

Batches:  83%|████████▎ | 10/12 [00:10<00:01,  1.31it/s]

Batches:  92%|█████████▏| 11/12 [00:11<00:00,  1.39it/s]

Batches: 100%|██████████| 12/12 [00:11<00:00,  1.63it/s]

Batches: 100%|██████████| 12/12 [00:11<00:00,  1.03it/s]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [13]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [14]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [15]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=EXTRACT_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [16]:
#@title 4.1 — Golden Dataset (chọn từ bộ 50 câu của giảng viên)
GOLDEN_PATH = f"{WORK_DIR}/golden_dataset.csv" if IN_COLAB else "data/golden_dataset.csv"

detailed = pd.read_csv(GOLDEN_DETAILED_PATH)
extracted_rows = set(extraction_source["row_idx"])

def evidence_coverage(x):
    """Tỷ lệ bài báo evidence của câu hỏi đã lọt vào tập extraction."""
    try:
        ids = set(json.loads(x))
        return len(ids & extracted_rows) / max(1, len(ids))
    except Exception:
        return 0.0

detailed["coverage"] = detailed["evidence_row_ids_0based"].map(evidence_coverage)
print("Coverage theo nhóm:")
display(detailed.groupby("group")["coverage"].describe()[["count", "mean", "min", "max"]])

# Chọn 12 câu (>= 5, phủ đủ 3 nhóm), ưu tiên câu có evidence coverage cao nhất
PER_GROUP = {"factoid": 4, "multi-hop": 4, "cross-doc": 4}
parts = []
for grp, n in PER_GROUP.items():
    g = detailed[detailed.group == grp].sort_values("coverage", ascending=False).head(n)
    parts.append(g)
golden_full = pd.concat(parts).reset_index(drop=True)
print("Coverage của các câu được chọn:",
      golden_full[["id", "group", "coverage"]].to_dict("records"))

golden_df = golden_full[["id", "group", "question", "reference_answer", "reference_evidence"]].copy()
Path(GOLDEN_PATH).parent.mkdir(parents=True, exist_ok=True)
golden_df.to_csv(GOLDEN_PATH, index=False)
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id", "group", "question", "reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required - set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id", "question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

validate_golden(golden_df)


Coverage theo nhóm:


,count,mean,min,max
group,,,,
cross-doc,22.0,1.0,1.0,1.0
factoid,5.0,1.0,1.0,1.0
multi-hop,23.0,1.0,1.0,1.0


Coverage của các câu được chọn: [{'id': 'G5000-03', 'group': 'factoid', 'coverage': 1.0}, {'id': 'G5000-13', 'group': 'factoid', 'coverage': 1.0}, {'id': 'G5000-24', 'group': 'factoid', 'coverage': 1.0}, {'id': 'G5000-41', 'group': 'factoid', 'coverage': 1.0}, {'id': 'G5000-01', 'group': 'multi-hop', 'coverage': 1.0}, {'id': 'G5000-05', 'group': 'multi-hop', 'coverage': 1.0}, {'id': 'G5000-06', 'group': 'multi-hop', 'coverage': 1.0}, {'id': 'G5000-08', 'group': 'multi-hop', 'coverage': 1.0}, {'id': 'G5000-02', 'group': 'cross-doc', 'coverage': 1.0}, {'id': 'G5000-04', 'group': 'cross-doc', 'coverage': 1.0}, {'id': 'G5000-07', 'group': 'cross-doc', 'coverage': 1.0}, {'id': 'G5000-09', 'group': 'cross-doc', 'coverage': 1.0}]


,id,group,question,reference_answer,reference_evidence
0,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935 (2023-01-18 22:37:00): A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular...
1,G5000-13,factoid,"Which three companies launched AI Lighthouse in the selected 5,000-row scope?","ServiceNow, NVIDIA, and Accenture.",row 648 (2023-07-27 18:43:00): ServiceNow NVIDIA and Accenture Partner to Accelerate Generative AI Adoption for Ente...
2,G5000-24,factoid,What Microsoft settlement amount is reported for illegally collecting children's personal information?,$20 million.,row 2511 (2023-06-07 09:36:00): Microsoft pays $20 million settlement for collecting children’s data
3,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ...",Axis Security; a unified Secure Access Service Edge (SASE) solution.,row 4762 (2023-03-02 22:32:00): Hewlett Packard Enterprise Fortifies Network Security With Acquisition of Security S...
4,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
5,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 935 (2023-01-18 22:37:00): A Leap in...
6,G5000-06,multi-hop,Trace ServiceNow's generative-AI product/partner evolution from May through July 2023: what partnership began in May...,"In May, ServiceNow partnered with NVIDIA to build enterprise-grade generative AI for workflow automation. In June, S...",row 746 (2023-05-17 18:58:00): ServiceNow and NVIDIA Announce Partnership to Build Generative AI Across Enterprise I...
7,G5000-08,multi-hop,"Which external organizations are connected to ServiceNow's generative-AI efforts in the selected data, and what dist...",NVIDIA is ServiceNow's generative-AI technology partner and later co-launch partner for AI Lighthouse; Accenture joi...,row 746 (2023-05-17 18:58:00): ServiceNow and NVIDIA Announce Partnership to Build Generative AI Across Enterprise I...
8,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
9,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...


✅ Golden Dataset valid.


In [17]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [18]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = f"{WORK_DIR}/graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
t0 = time.perf_counter()
eval_results_df = run_evaluation(golden_df)
print(f"Evaluation: {(time.perf_counter()-t0)/60:.1f} phút | Groq stats: {GROQ_STATS}")
display(eval_results_df)

✅ Golden Dataset valid.


Evaluation:   0%|          | 0/12 [00:00<?, ?it/s]

Evaluation:   8%|▊         | 1/12 [00:24<04:33, 24.90s/it]

Evaluation:  17%|█▋        | 2/12 [00:45<03:42, 22.29s/it]

Evaluation:  25%|██▌       | 3/12 [01:17<03:59, 26.67s/it]

...

Evaluation:  92%|█████████▏| 11/12 [05:36<00:34, 34.49s/it]

Evaluation: 100%|██████████| 12/12 [06:11<00:00, 34.60s/it]

Evaluation: 100%|██████████| 12/12 [06:11<00:00, 30.94s/it]

Evaluation: 6.2 phút | Groq stats: {'calls': 60, 'total_tokens': 49617}


,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.","The later connectivity report cited more than 100 million IoT devices, 9,000 enterprises, and 190 countries [chunk_i...","The later connectivity report cited more than 100 million IoT devices, 9,000 enterprises, and 190 countries [chunk_i...",5,5,5,5,5,5,2.127725,1.049556,674,784,"The candidate answer directly provides the three figures—more than 100 million IoT devices, 9,000 enterprises, and 1...","The candidate answer provides the exact figures for IoT devices, enterprises, and countries as stated in the referen...",0
1,G5000-13,factoid,"Which three companies launched AI Lighthouse in the selected 5,000-row scope?","ServiceNow, NVIDIA, and Accenture.","The three companies that launched AI Lighthouse are ServiceNow, NVIDIA, and Accenture [chunk_id=0304ea2aa9bd55fd04cc...","The three companies that launched AI Lighthouse are ServiceNow, NVIDIA, and Accenture [chunk_id=0304ea2aa9bd55fd04cc...",5,5,5,5,5,5,1.292684,1.295537,608,433,"The candidate correctly lists all three companies—ServiceNow, NVIDIA, and Accenture—as stated in the reference and s...","The candidate correctly lists all three companies (ServiceNow, NVIDIA, Accenture) exactly as in the reference answer...",0
2,G5000-24,factoid,What Microsoft settlement amount is reported for illegally collecting children's personal information?,$20 million.,Microsoft has agreed to pay a $20 million settlement for illegally collecting personal information from children wit...,Microsoft has agreed to pay a $20 million settlement for illegally collecting personal information from children wit...,5,5,5,5,5,5,1.597967,1.575298,696,1721,"The candidate directly states the settlement amount as $20 million, which exactly matches the reference answer. The ...","The candidate directly states the settlement amount as $20 million, which exactly matches the reference answer. It c...",0
3,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ...",Axis Security; a unified Secure Access Service Edge (SASE) solution.,"Hewlett Packard Enterprise (HPE) agreed to acquire Axis Security, a cloud security provider, to expand its edge-to-c...",HPE agreed to acquire Axis Security to expand its edge-to-cloud security capabilities. The deal will support a unifi...,5,5,5,5,5,5,2.619932,1.207340,767,650,The candidate correctly identifies Axis Security as the company HPE agreed to acquire and states that the deal suppo...,The candidate correctly identifies Axis Security as the company HPE will acquire and states that the deal supports a...,0
4,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...","Aeris acquired Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, along with related assets, as part...","Aeris acquired Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, along with related assets, as part...",5,5,5,5,5,5,1.946346,2.348457,725,832,The answer lists both Ericsson businesses (IoT Accelerator and Connected Vehicle Cloud) and correctly states the sca...,The answer lists both Ericsson businesses (IoT Accelerator and Connected Vehicle Cloud) and correctly states the sca...,0
5,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and 

In [19]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
OUT_DIR = "/content" if IN_COLAB else "outputs"
Path(OUT_DIR).mkdir(exist_ok=True)
eval_results_df.to_csv(f"{OUT_DIR}/graphrag_eval_results.csv", index=False)
comparison_df.to_csv(f"{OUT_DIR}/graphrag_vs_flatrag_summary.csv", index=False)
print("Exported:", f"{OUT_DIR}/graphrag_eval_results.csv", "|", f"{OUT_DIR}/graphrag_vs_flatrag_summary.csv")

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),5.759,3.035,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,768.000,867.750,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),1.910,1.282,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,686.250,897.000,Flat RAG thường rẻ/nhanh hơn.


Exported: outputs/graphrag_eval_results.csv | outputs/graphrag_vs_flatrag_summary.csv


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [20]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

def test_supernode_policy_synthetic():
    """Graph thật ở scale lab chưa có node degree > 100 — tạo hub giả 120 cạnh
    để chứng minh cơ chế cap hoạt động, xong xoá sạch."""
    run_cypher("""
    MERGE (x:Entity {id:'__synthetic_hub__'})
    SET x:Company, x.name='SyntheticHub', x.entity_type='Company', x.name_norm='synthetichub'
    """)
    rows = [{"i": str(i), "d": f"2024-01-{(i % 28) + 1:02d}"} for i in range(120)]
    run_cypher("""
    UNWIND $rows AS row
    MERGE (m:Entity {id: '__syn_' + row.i})
    SET m:Company, m.name = 'SynNode' + row.i, m.entity_type = 'Company'
    WITH m, row
    MATCH (x:Entity {id:'__synthetic_hub__'})
    MERGE (x)-[r:PARTNERED_WITH {source_chunk_id: 'synthetic::' + row.i}]->(m)
    SET r.published_date = row.d, r.evidence = 'synthetic edge', r.confidence = 1.0
    """, rows=rows)
    deg = node_degree('__synthetic_hub__')
    limit = SUPER_NODE_EDGE_CAP if deg > SUPER_NODE_DEGREE else 1000
    fetched = recent_edges('__synthetic_hub__', limit)
    print(f"Synthetic hub: degree={deg}, fetched={len(fetched)} (cap={SUPER_NODE_EDGE_CAP})")
    assert deg > SUPER_NODE_DEGREE and len(fetched) <= SUPER_NODE_EDGE_CAP
    run_cypher("MATCH (n:Entity) WHERE n.id STARTS WITH '__syn' DETACH DELETE n")
    print("✅ Super-node cap verified trên hub giả 120 cạnh — đã dọn sạch.")

test_supernode_policy()
test_supernode_policy_synthetic()
show_resolution_audit(entity_resolution_audit_df)

{'id': 'fb0f4df56fab164ec48722f0', 'name': 'Microsoft', 'degree': 17} fetched= 17


Synthetic hub: degree=120, fetched=50 (cap=50)


✅ Super-node cap verified trên hub giả 120 cạnh — đã dọn sạch.


,type,left,right,similarity,decision
9,Technology,AI service,AI services,0.967647,MERGE_VECTOR
5,Company,Houston,Houston Texas,0.945098,REJECT_GUARD
0,Company,L&T Technology Services Limited,L&T Technology Services,0.925773,MERGE_VECTOR
4,Company,Fidelity National Information Services Inc.,Fidelity National Information Services,0.924524,MERGE_VECTOR
6,Company,Stewart Information Services Corporation,Stewart Information Services,0.922792,MERGE_VECTOR
1,Company,L&T Technology Services Limited,L&T Technology Services Ltd.,0.890695,REJECT_THRESHOLD
8,Technology,ChatGPT technology,ChatGPT,0.876033,REJECT_THRESHOLD
7,Company,Synergy Quantum India,Synergy Quantum,0.867454,REJECT_THRESHOLD
2,Company,L&T Technology Services,L&T Technology Services Ltd.,0.866094,REJECT_THRESHOLD
11,Technology,AI cloud services,AI services,0.842450,REJECT_THRESHOLD


High-similarity rejected pairs:


,type,left,right,similarity,decision
5,Company,Houston,Houston Texas,0.945098,REJECT_GUARD


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [21]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

community_df = build_communities()
print("Số community:", community_df.community_id.nunique(), "| nodes gán nhãn:", len(community_df))
display(pd.DataFrame(run_cypher("""
MATCH (n:Entity) WHERE n.community_id IS NOT NULL
WITH n.community_id AS cid, count(*) AS size, collect(n.name)[0..5] AS sample
RETURN cid, size, sample ORDER BY size DESC LIMIT 10
""")))

Số community: 141 | nodes gán nhãn: 382


,cid,size,sample
0,0,25,"[Vestwell, Google, Associated Press, JPMorgan, KPMG]"
1,1,11,"[Accenture, Deloitte, Cohere, Amazon, NVIDIA]"
2,3,7,"[Thales, L&T Technology Services Ltd., Qualcomm, L&T Technology Services, Airbus]"
3,2,7,"[IBM, Amazon Web Services, Advanced Micro Devices Inc, Intel, 3D V-Cache Tech]"
4,4,5,"[Renovus, Aretum, Data Serv, Deep Water Point & Associates, Insight]"
5,5,5,"[Broadridge Financial Solutions, Syndio, Lumen, Kate Johnson, Syndio’s technology]"
6,6,4,"[Ericsson, Aeris, IoT Business from Ericsson, IoT Accelerator and Connected Vehicle Cloud businesses]"
7,10,4,"[Omantel, Oman Ministry of Transport Communication and Information Technology, Huawei, cloud infrastructure]"
8,7,4,"[H2O.ai, Snowflake, H2O AI Cloud, Data Management Services]"
9,14,4,"[DriveNets, Or Sadeh, Krayden, Yuval Lev]"


In [22]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# Demo định lượng: chạy self-correction trên 1 câu multi-hop
demo_q = golden_df[golden_df.group == "multi-hop"].iloc[0]
sc = self_correcting_context(demo_q.question)
print("Câu hỏi:", demo_q.question)
print("Route:", sc["route"])
print("Missing:", (sc["missing"] or "")[:300])
print("Tổng Groq usage cả phiên:", GROQ_STATS)


Câu hỏi: Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeris, and what scale of IoT connectivity was attributed to the resulting Aeris footprint?
Route: hop2
Missing: 
Tổng Groq usage cả phiên: {'calls': 62, 'total_tokens': 50185}


# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau